# Conversion of SMILES to Canonical_SMILES

In [ ]:
import pandas as pd
from rdkit import Chem

# Loading file
df = pd.read_csv("combined_descriptors_labels.csv", sep=",")

# Function to convert SMILES into canonical SMILES
def to_canonical(smile):
    try:
        mol = Chem.MolFromSmiles(smile)
        if mol:
            return Chem.MolToSmiles(mol, canonical=True)
        else:
            return None
    except:
        return None

df["Canonical_SMILES"] = df.iloc[:,0].apply(to_canonical)

df.to_csv("your_dataset_with_canonical.csv", index=False)

print(df.head())


# Extracting columns and cmbine

In [ ]:
import pandas as pd

# Loading both datasets
df1 = pd.read_csv("dataset_with_BBclass_modified_BBB_label.csv")
df2 = pd.read_csv("final_canonical_cleaned_B3DB.csv")

# Dataset 1 B3DB
df1_selected = df1[['Unnamed: 0', 'Canonical_SMILES', 'BBclass_modified', 'logBB']].copy()

# Set ALL compound names to Unknown
df1_selected['Unnamed: 0'] = "Unknown"

df1_selected.rename(columns={
    'Unnamed: 0': 'compound_name',
    'Canonical_SMILES': 'canonical_smiles',
    'BBclass_modified': 'BBclass',
    'logBB': 'logBB'
}, inplace=True)

#Dataset 2 logBBB
df2_selected = df2[['compound_name', 'canonical_smiles', 'BBB+/BBB-', 'logBB']].copy()
df2_selected.rename(columns={
    'BBB+/BBB-': 'BBclass',
    'logBB': 'logBB'
}, inplace=True)

combined_df = pd.concat([df1_selected, df2_selected], ignore_index=True)

combined_df.to_csv("combined_compound_SMILES_BBclass.csv", index=False)


# Data Cleaning after combining

In [ ]:
import pandas as pd
from rdkit import Chem

df = pd.read_csv("combined_compound_SMILES_BBclass_updated.csv")

# Remove SMILES > 512 characters
df = df[df['canonical_smiles'].str.len() <= 512]

# Remove duplicate SMILES
df = df.drop_duplicates(subset=['canonical_smiles'])

# Remove invalid SMILES using RDKit
def is_valid_smiles(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        return mol is not None
    except:
        return False

df = df[df['canonical_smiles'].apply(is_valid_smiles)]

df.to_csv("combined_compound_SMILES_BBclass_cleaned.csv", index=False)


# Compute Morgan Fingerprints

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

df = pd.read_csv("combined_compound_SMILES_BBclass_cleaned.csv")

# Function to compute Morgan Fingerprints
def morgan_fp(smiles, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [0] * nBits
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)
    return list(fp)

# Generate fingerprints for each SMILES
fps = df['canonical_smiles'].apply(morgan_fp)

fp_df = pd.DataFrame(fps.tolist(), columns=[f"fp_{i}" for i in range(2048)])

logBB_index = df.columns.get_loc("logBB") + 1

df_final = pd.concat([df.iloc[:, :logBB_index], fp_df, df.iloc[:, logBB_index:]], axis=1)

df_final.to_csv("combined_compound_SMILES_BBclass_with_FP.csv", index=False)